# 23N Field Tests #11 · Mapping a city flood from radar

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/23nlabs/field-tests/blob/ft11-flood-mapping-radar/ft11-flood-mapping-radar/ft11_flood_mapping_radar.ipynb)

**Question.** How fast can open radar data map a city flood? A case study of Vadodara in August 2024: how much
open floodwater could the first free satellite pass show, and how many buildings and roads were in it?

**Why it matters.** On 26 August 2024 the Vishwamitri river crossed its 26 ft danger mark in Vadodara at about
4 pm, after 12.44 inches of rain fell on the city in 24 hours; the official gauge then held at 35.25 ft for more
than a day (DeshGujarat timeline, 1 Oct 2024). Monsoon cloud hides a flood from optical satellites. Radar sees
through cloud, day and night, and the Copernicus Sentinel-1 radar data are free for anyone. ISRO's NRSC issued its
rapid flood map of the district from a Sentinel-1A pass of 28 August. This notebook asks what anyone can do with the
same open data and a free CPU.

**Method.** The first Sentinel-1 pass over Vadodara after the river crossed its danger mark is compared with a
dry-season baseline: the median of every pass from the **same orbit** in January–May 2024. A pixel is open
floodwater if it is dark like water in the flood image (threshold set by Otsu's method on that image) **and** at
least `CHANGE_DB` dB darker than its dry-season self. Water that is there in the dry season (lakes, the river
channel) is left out. The flood map is then overlaid on OpenStreetMap buildings and roads. Checks: the same detector
on a pass **before** the rain (it should find little), the **next** pass (the water should be receding), VH instead of
VV, and stricter and looser thresholds. No model is trained.

**Data and licences.** Copernicus Sentinel-1 GRD via the Microsoft Planetary Computer STAC API (Copernicus
Sentinel data terms: free, full and open, commercial use allowed; "Contains modified Copernicus Sentinel data 2024").
**No account and no key.** OpenStreetMap buildings, roads and the river line via the Overpass API (© OpenStreetMap
contributors, ODbL 1.0). Optional completeness check: Microsoft Global ML Building Footprints (CDLA Permissive 2.0).
No AI model weights are used.

**Runtime.** About 5–15 minutes on free Colab or Kaggle CPU: reading one small window from about 15 radar scenes,
three Overpass queries, and the video. No paid API keys.

*Independent study on public data by 23N AI Labs. Vadodara is used as a case study of a city flood: this says
nothing about any municipal body, dam authority or agency, and implies no affiliation with any of them. All areas
and counts are estimates from a stated method.*

## 1 · Setup

In [ ]:
# CELL: deps
# pystac-client finds the radar scenes, planetary-computer signs the download links (anonymous, free),
# rasterio reads only the window over the city from each Cloud-Optimised GeoTIFF.
import os, sys, subprocess
if os.environ.get("FT11_SMOKE_TEST", "0") != "1":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pystac-client", "planetary-computer", "rasterio",
                    "gradio"], check=False)

In [ ]:
# CELL: config
import os, sys, io, json, math, time, gzip, textwrap, subprocess, warnings, urllib.request, urllib.parse, datetime as dt
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from scipy import ndimage

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = os.path.exists("/kaggle/working")

# ---- Test switch. OFF by default. Only for build checks without internet: it replaces the radar and OpenStreetMap
# downloads with a synthetic city, river and flood on the same grid. Never post its output.
SMOKE_TEST = os.environ.get("FT11_SMOKE_TEST", "0") == "1"

FT_NUM = "11"

# ---- The event: documented facts (sources in the README), used to choose the passes and in the post
CITY = "Vadodara"
DANGER_MARK_FT = 26              # Vishwamitri danger mark at Kalaghoda bridge, ft
DANGER_CROSS_IST = "2024-08-26 16:00"   # the river crossed 26 ft at about 4 pm IST on 26 Aug 2024
RAIN_24H_IN = 12.44              # rain in Vadodara, 6 am 26 Aug to 6 am 27 Aug 2024, inches
PEAK_GAUGE_FT = 35.25            # official gauge, held for over 24 hours on 27-28 Aug
FACT_SOURCE = "DeshGujarat, 'Vishwamitri river flood in Vadodara in August 2024', 1 Oct 2024"
NRSC_NOTE = "NRSC/ISRO rapid flood map, Vadodara district, as on 28.08.2024, from a Sentinel-1A pass of 28-08-2024"
RAIN_START = "2024-08-24"        # the heavy-rain spell began on 24 Aug: the 'before' pass must be earlier
EVENT_MAX_DAYS = 6               # the flood pass is the first pass within 6 days after the danger mark
AFTER_END = "2024-09-30"         # the 'after' pass is the next same-orbit pass up to this date
DRY_WINDOW = ("2024-01-01", "2024-05-31")   # dry-season baseline (no monsoon rain)

# ---- Area: Vadodara city and its edges (our choice of box, not an administrative boundary)
AOI_BBOX = (73.10, 22.22, 73.28, 22.40)    # lon_min, lat_min, lon_max, lat_max
GRID_CRS = "EPSG:32643"          # UTM zone 43N (72-78° E)
GRID_RES_M = 10                  # Sentinel-1 GRD-H IW pixel spacing
READ_DECIMATION = 1              # read at full resolution (the window is small)

# ---- Radar data
STAC_URL = "https://planetarycomputer.microsoft.com/api/stac/v1"
COLLECTION = "sentinel-1-grd"
INSTRUMENT_MODE = "IW"
POLS = ("vv", "vh")
MAIN_POL = "vv"                  # VV separates open water from land best; VH is the robustness variant

# ---- Flood detector (named assumptions, printed with the result)
SPECKLE_PX = 5                   # 5x5 boxcar (50 m) against radar speckle
CHANGE_DB = 3.0                  # flood pixel must be ≥ 3 dB darker than its dry-season median
CHANGE_DB_ROBUST = (2.0, 4.0)    # looser and stricter variants
WATER_DB_OFFSET = 0.0            # added to the Otsu water threshold (0 = Otsu as is)
WATER_DB_ROBUST = (-1.0, 1.0)    # water threshold variants, dB
MIN_FLOOD_PX = 50                # patches smaller than 0.5 ha are dropped as speckle
MIN_COVER = 0.90                 # a pass is used only if it covers ≥ 90 % of the box
MIN_BASELINE = 4                 # at least 4 dry-season passes for the baseline median
DB_STEP = 0.4                    # radar stack stored as uint8 dB in 0.4 dB steps to save memory

# ---- Exposure
BUILDING_BUFFER_M = 20           # a building counts if water is within 20 m (radar can't see the ground beside walls)
ROAD_BUFFER_M = 10
RIVER_NEAR_KM = 2.0              # 'near the river' for the plausibility check
OVERPASS_URLS = ["https://overpass-api.de/api/interpreter", "https://overpass.kumi.systems/api/interpreter"]
ROAD_RE = "^(motorway|trunk|primary|secondary|tertiary|unclassified|residential|living_street|service)(_link)?$"
MAJOR_ROADS = ("motorway", "trunk", "primary", "secondary", "tertiary")
USE_MS_BUILDINGS = True          # optional cross-check of OSM completeness; skipped if the download fails
MS_INDEX_URLS = ["https://bfppub.blob.core.windows.net/%24web/2026-08-13/dataset-links.csv",
                 "https://minedbuildings.z5.web.core.windows.net/global-buildings/dataset-links.csv"]

# ---- Sanity-check bounds (what is already known)
MIN_FLOOD_KM2 = 0.5              # the river was 9+ ft over its danger mark for a day: a near-empty map is a failure
MAX_FLOOD_FRAC = 0.5             # more than half the box 'flooded' is a detector or calibration failure
MAX_PLACEBO_SHARE = 0.30         # the pre-rain pass must show < 30 % of the flood area
MIN_NEAR_RIVER_SHARE = 0.40      # most open floodwater should be within 2 km of the Vishwamitri
MAX_LAND_SHIFT_DB = 1.5          # unflooded land should not shift > 1.5 dB between baseline and flood pass
ROBUST_RATIO_MAX = 2.0           # area under the looser/stricter threshold must stay within ×2
MIN_WATER_SEP_DB = 4.0           # the water threshold must sit ≥ 4 dB below typical land
MAX_BLDG_FRAC = 0.6              # > 60 % of the city's buildings 'in water' is implausible for open-water mapping
MIN_OSM_BUILDINGS = 20000        # fewer mapped buildings in the box → OSM incomplete (note, not a failure)
OSM_VS_MS_MIN = 0.5              # OSM below half of Microsoft's count → building number is a lower bound (note)

# ---- Where things live (this branch until it is merged)
REPO_REF = "ft11-flood-mapping-radar"
RAW = f"https://raw.githubusercontent.com/23nlabs/field-tests/{REPO_REF}"

# ---- Brand kit
if not os.path.exists("brand23n.py"):
    urllib.request.urlretrieve(f"{RAW}/brand/brand23n.py", "brand23n.py")
import brand23n as B
B.setup()
DATA_CREDIT = "Contains modified Copernicus Sentinel data 2024, via Microsoft Planetary Computer · © OpenStreetMap contributors (ODbL)"
IST = pd.Timedelta(hours=5, minutes=30)
DANGER_CROSS_UTC = pd.Timestamp(DANGER_CROSS_IST) - IST

print("Colab:", IN_COLAB, "| Kaggle:", IN_KAGGLE, "| SMOKE_TEST:", SMOKE_TEST)
if SMOKE_TEST:
    print("!! SMOKE TEST MODE: synthetic radar, buildings and roads. These results are NOT real. Do NOT post.")

In [ ]:
# CELL: aoi
# A 10 m UTM grid over the box. Every radar pass is resampled onto it, so pixels line up across dates.
import rasterio
from rasterio.warp import transform as warp_transform
from affine import Affine

def to_utm(lons, lats):
    x, y = warp_transform("EPSG:4326", GRID_CRS, list(lons), list(lats))
    return np.array(x), np.array(y)

def to_lonlat(xs, ys):
    lon, lat = warp_transform(GRID_CRS, "EPSG:4326", list(xs), list(ys))
    return np.array(lon), np.array(lat)

lo0, la0, lo1, la1 = AOI_BBOX
_cx, _cy = to_utm([lo0, lo1, lo1, lo0], [la0, la0, la1, la1])
_x0 = math.floor(_cx.min() / GRID_RES_M) * GRID_RES_M
_y1 = math.ceil(_cy.max() / GRID_RES_M) * GRID_RES_M
W = int(math.ceil((_cx.max() - _x0) / GRID_RES_M)); H = int(math.ceil((_y1 - _cy.min()) / GRID_RES_M))
TR = Affine(GRID_RES_M, 0, _x0, 0, -GRID_RES_M, _y1)
_ex = np.r_[np.linspace(0, W, 30), np.full(30, W), np.linspace(W, 0, 30), np.zeros(30)]
_ey = np.r_[np.zeros(30), np.linspace(0, H, 30), np.full(30, H), np.linspace(H, 0, 30)]
_gx, _gy = TR * (_ex, _ey)
_elon, _elat = to_lonlat(_gx, _gy)
GRID = dict(H=H, W=W, transform=TR, edge_lon=_elon, edge_lat=_elat)
PIX_KM2 = GRID_RES_M ** 2 / 1e6
EXTENT_KM = [0, W * GRID_RES_M / 1000, H * GRID_RES_M / 1000, 0]

def lonlat_to_rc(lon, lat):
    x, y = to_utm(lon, lat)
    c, r = ~TR * (x, y)
    return np.asarray(r), np.asarray(c)

def disk(r_px):
    r = max(1, int(round(r_px))); yy, xx = np.mgrid[-r:r + 1, -r:r + 1]
    return (xx ** 2 + yy ** 2) <= r * r

print(f"Grid: {W} x {H} pixels at {GRID_RES_M} m = {W * H * PIX_KM2:.0f} km² ({GRID_CRS})")

## 2 · Data: Sentinel-1 radar passes, and OpenStreetMap buildings and roads

The Planetary Computer's STAC API lists every Sentinel-1 GRD scene. Each scene is a Cloud-Optimised GeoTIFF of about
1 GB; we read only the window over the city (a few MB). GRD images are in radar geometry with ground control points;
the reader uses those points to resample the window onto our 10 m grid. Values are uncalibrated digital numbers
squared: the detector uses only **differences in dB between passes from the same orbit** (same viewing geometry, same
calibration curve), and it checks that unflooded land does not shift between them.

Passes used, all from the orbit of the flood pass:
- **Flood pass:** the first pass after the river crossed its danger mark (within `EVENT_MAX_DAYS`).
- **Baseline:** every pass from January to May 2024 (dry season), combined as a median.
- **Before:** the last pass before the rain began (`RAIN_START`): a placebo test.
- **After:** the next pass after the flood pass: the water should be receding.

OpenStreetMap buildings, roads and the Vishwamitri river line come from the Overpass API (no key).

In [ ]:
# CELL: loaders
from rasterio.windows import Window
from rasterio.warp import reproject, Resampling
from rasterio.transform import GCPTransformer, rowcol as affine_rowcol
from rasterio.control import GroundControlPoint

def to_u8(inten):
    with np.errstate(divide="ignore", invalid="ignore"):
        db = 10 * np.log10(inten)
    q = np.clip(np.round(db / DB_STEP) + 1, 1, 255)
    return np.where(np.isfinite(db), q, 0).astype(np.uint8)

def from_u8(q):
    return np.where(q > 0, 10 ** (((q.astype(np.float32) - 1) * DB_STEP) / 10), np.nan).astype(np.float32)

def u8_db(q):
    return np.where(q > 0, (q.astype(np.float32) - 1) * DB_STEP, np.nan)

def read_grid_from_src(src, decim=READ_DECIMATION, pad=64, gcp_margin=3000):
    # Returns the grid (float32 intensity, NaN = no data) or None if the scene misses the box.
    gcps, gcrs = src.gcps
    use_gcps = bool(gcps) and (src.crs is None or src.transform.is_identity)
    if use_gcps:
        rows, cols = GCPTransformer(gcps).rowcol(GRID["edge_lon"], GRID["edge_lat"])
    else:
        x, y = warp_transform("EPSG:4326", src.crs, list(GRID["edge_lon"]), list(GRID["edge_lat"]))
        rows, cols = affine_rowcol(src.transform, x, y)
    rows, cols = np.asarray(rows), np.asarray(cols)
    r0, r1 = max(0, int(rows.min()) - pad), min(src.height, int(rows.max()) + pad)
    c0, c1 = max(0, int(cols.min()) - pad), min(src.width, int(cols.max()) + pad)
    h, w = (r1 - r0) // decim, (c1 - c0) // decim
    if h < 16 or w < 16:
        return None
    win = Window(c0, r0, w * decim, h * decim)
    dn = src.read(1, window=win, out_shape=(h, w), resampling=Resampling.average).astype(np.float32)
    inten = np.where(dn > 0, dn * dn, np.nan).astype(np.float32)
    if not np.isfinite(inten).any():
        return None
    dst = np.full((GRID["H"], GRID["W"]), np.nan, np.float32)
    if use_gcps:
        sel = [g for g in gcps if r0 - gcp_margin <= g.row <= r1 + gcp_margin and c0 - gcp_margin <= g.col <= c1 + gcp_margin]
        if len(sel) < 6:
            sel = list(gcps)
        loc = [GroundControlPoint(row=(g.row - r0) / decim, col=(g.col - c0) / decim, x=g.x, y=g.y, z=g.z) for g in sel]
        reproject(inten, dst, gcps=loc, src_crs=gcrs, src_nodata=np.nan, dst_transform=GRID["transform"],
                  dst_crs=GRID_CRS, dst_nodata=np.nan, resampling=Resampling.bilinear)
    else:
        wtr = src.window_transform(win) * Affine.scale(decim)
        reproject(inten, dst, src_transform=wtr, src_crs=src.crs, src_nodata=np.nan, dst_transform=GRID["transform"],
                  dst_crs=GRID_CRS, dst_nodata=np.nan, resampling=Resampling.bilinear)
    return dst

GDAL_ENV = dict(GDAL_DISABLE_READDIR_ON_OPEN="EMPTY_DIR", GDAL_HTTP_MAX_RETRY="4", GDAL_HTTP_RETRY_DELAY="2",
                GDAL_HTTP_MULTIRANGE="YES", GDAL_HTTP_MERGE_CONSECUTIVE_RANGES="YES", VSI_CACHE="TRUE")

def read_href(href, tries=3):
    import planetary_computer as pc
    last = None
    for k in range(tries):
        try:
            with rasterio.Env(**GDAL_ENV):
                with rasterio.open(pc.sign(href)) as src:      # signed just before use: tokens expire
                    return read_grid_from_src(src)
        except Exception as e:
            last = e; time.sleep(2 * (k + 1))
    raise last

def read_pass(row):
    # Mosaics the slices of one pass, per polarisation. Returns ({pol: uint8 grid}, coverage, errors).
    out, errs = {}, []
    for pol in POLS:
        acc = None
        for href in row[f"hrefs_{pol}"]:
            try:
                g = read_href(href)
            except Exception as e:
                errs.append(f"{pol}: {type(e).__name__}: {str(e)[:120]}"); continue
            if g is not None:
                acc = g if acc is None else np.where(np.isfinite(acc), acc, g)
        out[pol] = acc
    cov = float(np.isfinite(out[MAIN_POL]).mean()) if out.get(MAIN_POL) is not None else 0.0
    return {p: (to_u8(a) if a is not None else np.zeros((GRID["H"], GRID["W"]), np.uint8)) for p, a in out.items()}, cov, errs

def utc_naive(t):
    t = pd.Timestamp(t)
    return t.tz_convert("UTC").tz_localize(None) if t.tzinfo else t

def search_passes():
    import pystac_client
    cat = pystac_client.Client.open(STAC_URL)
    s = cat.search(collections=[COLLECTION], bbox=AOI_BBOX, datetime=f"{DRY_WINDOW[0]}/{AFTER_END}",
                   query={"sar:instrument_mode": {"eq": INSTRUMENT_MODE}})
    rows = []
    for it in s.items():
        if not all(p in it.assets for p in POLS):
            continue
        p = it.properties
        rows.append(dict(id=it.id, time=utc_naive(it.datetime), platform=str(p.get("platform", "?")).upper(),
                         orbit=str(p.get("sat:orbit_state", "?")).lower(), rel_orbit=p.get("sat:relative_orbit"),
                         **{f"href_{pol}": it.assets[pol].href for pol in POLS}))
    items = pd.DataFrame(rows)
    if items.empty:
        return items
    # Same orbit = same relative orbit number; if the catalogue lacks it, the same orbit direction and time of day.
    ro = items["rel_orbit"].map(lambda v: "" if v is None or (isinstance(v, float) and np.isnan(v)) else str(int(v)))
    tod = items["time"].dt.hour.astype(str)
    items["track"] = items["orbit"] + "|" + np.where(ro != "", "rel" + ro, "h" + tod)
    items["pass_key"] = items["platform"] + "|" + items["track"] + "|" + items["time"].dt.strftime("%Y-%m-%d")
    agg = dict(time=("time", "min"), platform=("platform", "first"), orbit=("orbit", "first"),
               track=("track", "first"), n_slices=("id", "size"))
    agg.update({f"hrefs_{pol}": (f"href_{pol}", list) for pol in POLS})
    return items.sort_values("time").groupby("pass_key").agg(**agg).sort_values("time").reset_index()

def load_real():
    allp = search_passes()
    if allp.empty:
        raise RuntimeError("STAC search returned no Sentinel-1 IW scenes over the box")
    print(f"STAC: {len(allp)} passes over the box between {DRY_WINDOW[0]} and {AFTER_END}")
    window = allp[(allp.time > DANGER_CROSS_UTC) & (allp.time <= DANGER_CROSS_UTC + pd.Timedelta(days=EVENT_MAX_DAYS))]
    print("Passes in the flood window:"); print(window[["time", "platform", "orbit", "track", "n_slices"]].to_string(index=False))
    chosen, errors = [], []
    ev = None
    for _, r in window.iterrows():                       # first pass that covers the box
        q, cov, errs = read_pass(r); errors += errs
        print(f"  {r.time:%Y-%m-%d %H:%M} UTC {r.track}: covers {cov:.0%} of the box")
        if cov >= MIN_COVER:
            ev = r; chosen.append(("event", r, q, cov)); break
    if ev is None:
        raise RuntimeError("no Sentinel-1 pass covered the box in the flood window")
    same = allp[allp.track == ev.track]
    dry = same[(same.time >= DRY_WINDOW[0]) & (same.time <= pd.Timestamp(DRY_WINDOW[1]) + pd.Timedelta(days=1))]
    pre = same[(same.time < pd.Timestamp(RAIN_START)) & (same.time > pd.Timestamp(RAIN_START) - pd.Timedelta(days=40))].tail(1)
    aft = same[(same.time > ev.time + pd.Timedelta(days=1)) & (same.time <= pd.Timestamp(AFTER_END) + pd.Timedelta(days=1))].head(1)
    jobs = [("dry", r) for _, r in dry.iterrows()] + [("pre", r) for _, r in pre.iterrows()] + [("after", r) for _, r in aft.iterrows()]
    print(f"Same orbit ({ev.track}): {len(dry)} dry-season passes, {len(pre)} before, {len(aft)} after")
    with ThreadPoolExecutor(4) as ex:
        for (role, r), (q, cov, errs) in zip(jobs, ex.map(lambda j: read_pass(j[1]), jobs)):
            errors += errs
            if cov >= MIN_COVER:
                chosen.append((role, r, q, cov))
            else:
                print(f"  skipped {role} pass {r.time:%Y-%m-%d}: covers {cov:.0%} of the box")
    passes = pd.DataFrame([dict(role=role, time=r.time, platform=r.platform, orbit=r.orbit, track=r.track, cover=cov)
                           for role, r, q, cov in chosen])
    stack = {pol: np.stack([q[pol] for _, _, q, _ in chosen]) for pol in POLS}
    return passes, stack, window[["time", "platform", "orbit", "track"]].reset_index(drop=True), errors

# ---- OpenStreetMap via Overpass (no key). Cached to JSON so a re-run doesn't query again.
def overpass(query, cache):
    if os.path.exists(cache):
        return json.load(open(cache))
    last = None
    for url in OVERPASS_URLS:
        try:
            req = urllib.request.Request(url, data=urllib.parse.urlencode({"data": query}).encode(),
                                         headers={"User-Agent": "23N-Field-Tests/11 (+https://23nlabs.com)"})
            with urllib.request.urlopen(req, timeout=400) as r:
                js = json.load(r)
            json.dump(js, open(cache, "w"))
            return js
        except Exception as e:
            last = e; print(f"  Overpass {url} failed: {type(e).__name__} {str(e)[:100]}"); time.sleep(5)
    raise last

def load_osm():
    bb = f"{la0},{lo0},{la1},{lo1}"
    b = overpass(f'[out:json][timeout:300];(way["building"]({bb});relation["building"]({bb}););out center qt;',
                 "ft11_osm_buildings.json")
    pts = [(e["center"]["lon"], e["center"]["lat"]) for e in b["elements"] if "center" in e]
    rd = overpass(f'[out:json][timeout:300];way["highway"~"{ROAD_RE}"]({bb});out geom qt;', "ft11_osm_roads.json")
    roads = [(np.array([g["lon"] for g in e["geometry"]]), np.array([g["lat"] for g in e["geometry"]]),
              e.get("tags", {}).get("highway", "")) for e in rd["elements"] if len(e.get("geometry", [])) > 1]
    rv = overpass(f'[out:json][timeout:120];way["waterway"~"^(river|stream|canal)$"]["name"~"Vish[wv]amitri",i]({bb});out geom;',
                  "ft11_osm_river.json")
    river = [(np.array([g["lon"] for g in e["geometry"]]), np.array([g["lat"] for g in e["geometry"]]))
             for e in rv["elements"] if len(e.get("geometry", [])) > 1]
    stamp = b.get("osm3s", {}).get("timestamp_osm_base", "?")
    print(f"OSM (data as of {stamp}): {len(pts):,} buildings, {len(roads):,} road ways, {len(river)} Vishwamitri ways")
    return dict(bld=np.array(pts).reshape(-1, 2), roads=roads, river=river, stamp=stamp)

# ---- Optional: Microsoft Global ML Building Footprints, to check how complete OSM is here
def quadkey(lon, lat, z=9):
    s = math.sin(math.radians(lat)); x = (lon + 180) / 360; y = 0.5 - math.log((1 + s) / (1 - s)) / (4 * math.pi)
    n = 2 ** z; tx, ty = min(n - 1, int(x * n)), min(n - 1, int(y * n)); qk = ""
    for i in range(z, 0, -1):
        m = 1 << (i - 1); qk += str((1 if tx & m else 0) + (2 if ty & m else 0))
    return qk

def load_ms_buildings():
    qks = {quadkey(lo, la) for lo in (lo0, lo1) for la in (la0, la1)}
    idx = None
    for u in MS_INDEX_URLS:
        try:
            idx = pd.read_csv(u, dtype=str); break
        except Exception as e:
            print(f"  MS index {u} failed: {type(e).__name__}")
    if idx is None:
        return None
    cols = {c.lower(): c for c in idx.columns}
    loc, qkc, urlc = cols.get("location"), cols.get("quadkey"), cols.get("url")
    sel = idx[(idx[loc] == "India") & idx[qkc].isin(qks)]
    pts = []
    for u in sel[urlc]:
        fn = "ft11_ms_" + u.rsplit("/", 1)[-1]
        if not os.path.exists(fn):
            urllib.request.urlretrieve(u, fn)
        with gzip.open(fn, "rt") as f:
            for line in f:
                g = json.loads(line); g = g.get("geometry", g)
                ring = np.array(g["coordinates"][0] if g["type"] == "Polygon" else g["coordinates"][0][0])
                lon, lat = ring[:, 0].mean(), ring[:, 1].mean()
                if lo0 <= lon <= lo1 and la0 <= lat <= la1:
                    pts.append((lon, lat))
    print(f"Microsoft footprints: {len(pts):,} buildings in the box ({len(sel)} tile file(s))")
    return np.array(pts).reshape(-1, 2)

def synthetic_data(seed=11):
    # SMOKE TEST ONLY: a speckled radar scene with a bright city, fields, a lake and a winding river; a flood band
    # along the river on the flood pass, a few wet fields before the rain, and a smaller band after. Not real data.
    rng = np.random.default_rng(seed)
    rr, cc = np.mgrid[0:H, 0:W].astype(np.float32)
    river_c = W * 0.45 + W * 0.05 * np.sin(rr / H * 6.0)
    dist = np.abs(cc - river_c)
    urban = ((rr - H * 0.5) ** 2 + (cc - W * 0.55) ** 2) < (0.32 * min(H, W)) ** 2
    lake = ((rr - H * 0.3) ** 2 + (cc - W * 0.7) ** 2) < 25 ** 2
    channel = dist < 3
    flood_ev = (dist < 70 + 40 * np.sin(rr / 90.0)) & ~channel
    flood_ev |= ((rr - H * 0.8) ** 2 + (cc - W * 0.2) ** 2) < 120 ** 2
    flood_af = (dist < 20) & ~channel
    wet_pre = ((rr - H * 0.15) ** 2 + (cc - W * 0.15) ** 2) < 30 ** 2
    base = np.where(urban, 33.0, 27.0) + 1.5 * np.sin(cc / 37.0) * np.cos(rr / 53.0)
    base = np.where(channel | lake, 15.0, base)
    def scene(water):
        db = np.where(water, 15.0, base)
        return to_u8(10 ** (db / 10) * rng.gamma(4.4, 1 / 4.4, (H, W)).astype(np.float32))
    rows, qs = [], []
    for d in pd.date_range("2024-01-08", "2024-05-31", freq="12D"):
        rows.append(("dry", d + pd.Timedelta(hours=1, minutes=4))); qs.append(None)
    rows += [("event", pd.Timestamp("2024-08-28 01:04")), ("pre", pd.Timestamp("2024-08-16 01:04")),
             ("after", pd.Timestamp("2024-09-09 01:04"))]
    water = {"dry": np.zeros((H, W), bool), "event": flood_ev, "pre": wet_pre, "after": flood_af}
    stack = {pol: np.stack([scene(water[r]) if pol == "vv" else to_u8(from_u8(scene(water[r])) / 5) for r, _ in rows])
             for pol in POLS}
    passes = pd.DataFrame([dict(role=r, time=t, platform="SYNTHETIC", orbit="descending", track="descending|synthetic",
                                cover=1.0) for r, t in rows])
    window = passes[passes.role == "event"][["time", "platform", "orbit", "track"]].reset_index(drop=True)
    # synthetic OSM: buildings dense in the city, a road grid, and the river line
    n = 60000
    br = np.r_[rng.normal(H * 0.5, H * 0.18, int(n * 0.8)), rng.uniform(0, H, int(n * 0.2))]
    bc = np.r_[rng.normal(W * 0.55, W * 0.18, int(n * 0.8)), rng.uniform(0, W, int(n * 0.2))]
    ok = (br >= 0) & (br < H) & (bc >= 0) & (bc < W)
    bx, by = TR * (bc[ok], br[ok]); blon, blat = to_lonlat(bx, by)
    roads = []
    for k, f in enumerate(np.linspace(0.05, 0.95, 12)):
        for horiz in (True, False):
            t = np.linspace(0, 1, 40)
            c_, r_ = (t * W, np.full_like(t, f * H)) if horiz else (np.full_like(t, f * W), t * H)
            x_, y_ = TR * (c_, r_); lo_, la_ = to_lonlat(x_, y_)
            roads.append((lo_, la_, "primary" if k % 4 == 0 else "residential"))
    rx, ry = TR * (river_c[::20, 0], rr[::20, 0]); rlon, rlat = to_lonlat(rx, ry)
    osm = dict(bld=np.c_[blon, blat], roads=roads, river=[(rlon, rlat)], stamp="SYNTHETIC")
    return passes, stack, window, [], osm

In [ ]:
# CELL: load_data
CACHE = f"ft{FT_NUM}_cache.npz"
MS_BLD = None
if SMOKE_TEST:
    PASSES, STACK, WINDOW, READ_ERRORS, OSM = synthetic_data()
else:
    if os.path.exists(CACHE):
        _c = np.load(CACHE, allow_pickle=False)
        STACK = {pol: _c[pol] for pol in POLS}
        PASSES = pd.read_csv(CACHE.replace(".npz", "_passes.csv"), parse_dates=["time"])
        WINDOW = pd.read_csv(CACHE.replace(".npz", "_window.csv"), parse_dates=["time"]); READ_ERRORS = []
        print("Loaded the radar stack from the local cache (delete", CACHE, "to download again).")
    else:
        PASSES, STACK, WINDOW, READ_ERRORS = load_real()
        np.savez_compressed(CACHE, **STACK)
        PASSES.to_csv(CACHE.replace(".npz", "_passes.csv"), index=False)
        WINDOW.to_csv(CACHE.replace(".npz", "_window.csv"), index=False)
    OSM = load_osm()
    if USE_MS_BUILDINGS:
        try:
            MS_BLD = load_ms_buildings()
        except Exception as e:
            print("Microsoft footprints skipped:", type(e).__name__, str(e)[:120])

print(PASSES.assign(time=PASSES.time.dt.strftime("%Y-%m-%d %H:%M UTC")).to_string())
if READ_ERRORS:
    print(f"{len(READ_ERRORS)} read errors (first 3):", READ_ERRORS[:3])
I_EVENT = int(np.where(PASSES.role == "event")[0][0])
I_DRY = np.where(PASSES.role == "dry")[0]
I_PRE = int(np.where(PASSES.role == "pre")[0][0]) if (PASSES.role == "pre").any() else None
I_AFTER = int(np.where(PASSES.role == "after")[0][0]) if (PASSES.role == "after").any() else None
T_EVENT = PASSES.time.iloc[I_EVENT]
HOURS_AFTER = (T_EVENT - DANGER_CROSS_UTC) / pd.Timedelta(hours=1)
print(f"Flood pass: {T_EVENT:%d %b %Y %H:%M} UTC ({(T_EVENT + IST):%d %b %H:%M} IST), "
      f"{HOURS_AFTER:.0f} h after the danger mark · {len(I_DRY)} dry-season passes in the baseline")

## 3 · Method: dark now, and much darker than in the dry season

Calm open water reflects the radar pulse away from the satellite, so it is **dark**. Fields and city are brighter.

1. **Speckle.** Each pass is smoothed with a 5×5 (50 m) boxcar; the baseline is the median of all dry-season passes.
2. **Water threshold.** Otsu's method splits the flood pass's histogram (VV, dB) into dark and bright.
3. **Flood pixel.** Dark in the flood pass **and** at least `CHANGE_DB` dB darker than the dry-season baseline.
   Pixels that are dark in the dry season too (lakes, the river channel, smooth tarmac) are **permanent** and left
   out. Patches under 0.5 ha are dropped.
4. **Exposure.** A building counts if open floodwater is within `BUILDING_BUFFER_M` of its centre (radar often
   can't see the ground right beside a wall). Road length counts where the road runs through water.

Known blind spot: water in narrow streets between buildings often looks **bright** to radar (the pulse bounces off
water and wall back to the satellite), so open-water mapping undercounts dense urban flooding. That is the main
limitation, and why the number is "open floodwater", not "flooded area".

In [ ]:
# CELL: flood
def nan_smooth(a, n):
    ok = np.isfinite(a)
    num = ndimage.uniform_filter(np.where(ok, a, 0).astype(np.float32), n)
    den = ndimage.uniform_filter(ok.astype(np.float32), n)
    with np.errstate(divide="ignore", invalid="ignore"):
        out = num / den
    out[~ok | (den < 0.5)] = np.nan
    return out

def pass_db(pol, i):
    with np.errstate(divide="ignore", invalid="ignore"):
        return 10 * np.log10(nan_smooth(from_u8(STACK[pol][i]), SPECKLE_PX))

def median_db(pol, idx, chunk=128):
    out = np.full((H, W), np.nan, np.float32)
    for r in range(0, H, chunk):
        blk = u8_db(STACK[pol][idx, r:r + chunk])
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", RuntimeWarning)
            out[r:r + chunk] = np.nanmedian(blk, axis=0)
    return nan_smooth(out, SPECKLE_PX)

def otsu(vals, bins=256):
    vals = vals[np.isfinite(vals)]
    hist, edges = np.histogram(vals, bins=bins)
    mids = (edges[:-1] + edges[1:]) / 2; w = hist.cumsum(); m = (hist * mids).cumsum()
    wt, mt = w[-1], m[-1]
    with np.errstate(divide="ignore", invalid="ignore"):
        between = (mt * w - m * wt) ** 2 / (w * (wt - w))
    return float(mids[np.nanargmax(np.where(np.isfinite(between), between, -1))])

def detect(ev_db, base_db, change_db, water_t):
    valid = np.isfinite(ev_db) & np.isfinite(base_db)
    perm = valid & (base_db < water_t)
    fl = valid & (ev_db < water_t) & ((ev_db - base_db) < -change_db) & ~perm
    lab, n = ndimage.label(fl)
    if n:
        sizes = np.bincount(lab.ravel()); keep = sizes >= MIN_FLOOD_PX; keep[0] = False
        fl = keep[lab]
    return fl, perm, valid

if len(I_DRY) < 1:
    raise RuntimeError("no dry-season baseline pass: cannot run change detection")
BASE = {pol: median_db(pol, I_DRY) for pol in POLS}
EV = {pol: pass_db(pol, I_EVENT) for pol in POLS}
WATER_T = {pol: otsu(EV[pol]) + WATER_DB_OFFSET for pol in POLS}
FLOOD, PERM, VALID = detect(EV[MAIN_POL], BASE[MAIN_POL], CHANGE_DB, WATER_T[MAIN_POL])
AREA_KM2 = FLOOD.sum() * PIX_KM2
VALID_KM2 = VALID.sum() * PIX_KM2
LAND = VALID & ~FLOOD & ~PERM
LAND_DB = float(np.nanmedian(EV[MAIN_POL][LAND])) if LAND.any() else np.nan
WATER_SEP_DB = LAND_DB - WATER_T[MAIN_POL]
LAND_SHIFT_DB = float(np.nanmedian((EV[MAIN_POL] - BASE[MAIN_POL])[LAND])) if LAND.any() else np.nan

# robustness variants
VARIANTS = [("headline", f"{MAIN_POL.upper()}, {CHANGE_DB:g} dB, Otsu", AREA_KM2)]
for cdb in CHANGE_DB_ROBUST:
    VARIANTS.append(("change", f"{MAIN_POL.upper()}, {cdb:g} dB change", detect(EV[MAIN_POL], BASE[MAIN_POL], cdb, WATER_T[MAIN_POL])[0].sum() * PIX_KM2))
for off in WATER_DB_ROBUST:
    VARIANTS.append(("water", f"water threshold {off:+g} dB", detect(EV[MAIN_POL], BASE[MAIN_POL], CHANGE_DB, WATER_T[MAIN_POL] + off)[0].sum() * PIX_KM2))
_alt = [p for p in POLS if p != MAIN_POL][0]
VARIANTS.append(("pol", f"{_alt.upper()} instead of {MAIN_POL.upper()}", detect(EV[_alt], BASE[_alt], CHANGE_DB, WATER_T[_alt])[0].sum() * PIX_KM2))
PRE_DB = pass_db(MAIN_POL, I_PRE) if I_PRE is not None else None
if PRE_DB is not None:
    VARIANTS.append(("baseline", "baseline = the pass before the rain", detect(EV[MAIN_POL], PRE_DB, CHANGE_DB, WATER_T[MAIN_POL])[0].sum() * PIX_KM2))
VARIANTS = pd.DataFrame(VARIANTS, columns=["kind", "variant", "area_km2"])

# placebo (before the rain) and recession (next pass), same detector and same threshold
AREA_PRE = detect(PRE_DB, BASE[MAIN_POL], CHANGE_DB, WATER_T[MAIN_POL])[0].sum() * PIX_KM2 if PRE_DB is not None else np.nan
AFTER_DB = pass_db(MAIN_POL, I_AFTER) if I_AFTER is not None else None
FLOOD_AFTER = detect(AFTER_DB, BASE[MAIN_POL], CHANGE_DB, WATER_T[MAIN_POL])[0] if AFTER_DB is not None else None
AREA_AFTER = FLOOD_AFTER.sum() * PIX_KM2 if FLOOD_AFTER is not None else np.nan

print(f"Water threshold (Otsu, {MAIN_POL.upper()}): {WATER_T[MAIN_POL]:.1f} dB uncalibrated; typical land {LAND_DB:.1f} dB "
      f"({WATER_SEP_DB:.1f} dB apart)")
print(f"Unflooded land shift, flood pass vs baseline: {LAND_SHIFT_DB:+.2f} dB")
print(f"Open floodwater: {AREA_KM2:.2f} km² of {VALID_KM2:.0f} km² seen · permanent water {PERM.sum() * PIX_KM2:.2f} km²")
print(f"Before the rain: {AREA_PRE:.2f} km² · next pass: {AREA_AFTER:.2f} km²")
print(VARIANTS.round(2).to_string(index=False))

In [ ]:
# CELL: exposure
def points_in(mask, lon, lat):
    if len(lon) == 0:
        return np.zeros(0, bool), np.zeros(0, bool)
    r, c = lonlat_to_rc(lon, lat)
    r, c = np.floor(r).astype(int), np.floor(c).astype(int)
    inside = (r >= 0) & (r < H) & (c >= 0) & (c < W)
    hit = np.zeros(len(lon), bool)
    hit[inside] = mask[r[inside], c[inside]]
    seen = np.zeros(len(lon), bool); seen[inside] = VALID[r[inside], c[inside]]
    return hit, seen

FLOOD_B = ndimage.binary_dilation(FLOOD, structure=disk(BUILDING_BUFFER_M / GRID_RES_M))
BLD_HIT, BLD_SEEN = points_in(FLOOD_B, OSM["bld"][:, 0], OSM["bld"][:, 1])
N_BLD, N_BLD_SEEN = int(BLD_HIT.sum()), int(BLD_SEEN.sum())

def road_lengths(mask):
    # Samples every road every ~5 m; returns km in the box, km in water, km of major roads in water.
    tot = wet = wet_major = 0.0
    for lon, lat, hw in OSM["roads"]:
        x, y = to_utm(lon, lat)
        seg = np.hypot(np.diff(x), np.diff(y)); n = np.maximum(1, np.ceil(seg / 5).astype(int))
        t = np.concatenate([(np.arange(k) + 0.5) / k for k in n]); i = np.repeat(np.arange(len(seg)), n)
        px, py = x[i] + t * np.diff(x)[i], y[i] + t * np.diff(y)[i]; L = np.repeat(seg / n, n)
        c, r = ~TR * (px, py); r, c = np.floor(r).astype(int), np.floor(c).astype(int)
        ok = (r >= 0) & (r < H) & (c >= 0) & (c < W)
        ok[ok] &= VALID[r[ok], c[ok]]
        w = np.zeros(len(L), bool); w[ok] = mask[r[ok], c[ok]]
        tot += L[ok].sum(); wet += L[w].sum()
        if hw.replace("_link", "") in MAJOR_ROADS:
            wet_major += L[w].sum()
    return tot / 1000, wet / 1000, wet_major / 1000

ROAD_KM_TOTAL, ROAD_KM, ROAD_KM_MAJOR = road_lengths(ndimage.binary_dilation(FLOOD, structure=disk(ROAD_BUFFER_M / GRID_RES_M)))

# distance of every flood pixel from the Vishwamitri (OSM river line)
RIVER_MASK = np.zeros((H, W), bool)
for lon, lat in OSM["river"]:
    x, y = to_utm(lon, lat)
    n = np.maximum(1, np.ceil(np.hypot(np.diff(x), np.diff(y)) / 5).astype(int))
    px = np.concatenate([np.linspace(x[k], x[k + 1], n[k]) for k in range(len(n))])
    py = np.concatenate([np.linspace(y[k], y[k + 1], n[k]) for k in range(len(n))])
    c, r = ~TR * (px, py); r, c = np.floor(r).astype(int), np.floor(c).astype(int)
    ok = (r >= 0) & (r < H) & (c >= 0) & (c < W); RIVER_MASK[r[ok], c[ok]] = True
if RIVER_MASK.any():
    RIVER_DIST_KM = ndimage.distance_transform_edt(~RIVER_MASK) * GRID_RES_M / 1000
    NEAR_RIVER_SHARE = float((RIVER_DIST_KM[FLOOD] <= RIVER_NEAR_KM).mean()) if FLOOD.any() else np.nan
else:
    RIVER_DIST_KM, NEAR_RIVER_SHARE = None, np.nan

if MS_BLD is not None and len(MS_BLD):
    _h, _s = points_in(FLOOD_B, MS_BLD[:, 0], MS_BLD[:, 1]); N_MS, N_MS_SEEN = int(_h.sum()), int(_s.sum())
else:
    N_MS = N_MS_SEEN = None

print(f"OSM buildings: {N_BLD:,} of {N_BLD_SEEN:,} in the box have open floodwater within {BUILDING_BUFFER_M} m")
if N_MS is not None:
    print(f"Microsoft footprints: {N_MS:,} of {N_MS_SEEN:,}")
print(f"Roads: {ROAD_KM:.1f} km of {ROAD_KM_TOTAL:.0f} km run through water ({ROAD_KM_MAJOR:.1f} km major roads)")
print(f"Share of floodwater within {RIVER_NEAR_KM:g} km of the Vishwamitri: {NEAR_RIVER_SHARE:.0%}" if NEAR_RIVER_SHARE == NEAR_RIVER_SHARE
      else "Vishwamitri line not found in OSM: river-distance check skipped")

## 4 · Results: the headline, after the sanity checks

In [ ]:
# CELL: results
def fmt_km2(a):
    return f"{a:.1f}" if a == a else "n/a"

PROBLEMS, NOTES = [], []
if len(I_DRY) < MIN_BASELINE:
    PROBLEMS.append(f"only {len(I_DRY)} dry-season passes in the baseline (< {MIN_BASELINE}): baseline is noisy")
if not (0 < HOURS_AFTER <= EVENT_MAX_DAYS * 24):
    PROBLEMS.append(f"flood pass is {HOURS_AFTER:.0f} h from the danger mark: outside the flood window")
if AREA_KM2 < MIN_FLOOD_KM2:
    PROBLEMS.append(f"only {AREA_KM2:.2f} km² of floodwater found (< {MIN_FLOOD_KM2}): the river was 9+ ft over its danger mark, so the detector or the pass is wrong")
if VALID_KM2 and AREA_KM2 / VALID_KM2 > MAX_FLOOD_FRAC:
    PROBLEMS.append(f"{AREA_KM2 / VALID_KM2:.0%} of the box 'flooded' (> {MAX_FLOOD_FRAC:.0%}): detector or calibration failure")
if not (abs(LAND_SHIFT_DB) <= MAX_LAND_SHIFT_DB):
    PROBLEMS.append(f"unflooded land shifted {LAND_SHIFT_DB:+.1f} dB vs the baseline (> {MAX_LAND_SHIFT_DB}): calibration or processing break between passes")
if not (WATER_SEP_DB >= MIN_WATER_SEP_DB):
    PROBLEMS.append(f"water threshold only {WATER_SEP_DB:.1f} dB below typical land (< {MIN_WATER_SEP_DB}): Otsu split land types, not water from land")
if I_PRE is None:
    NOTES.append("no pass before the rain on the same orbit: placebo test skipped")
elif AREA_KM2 > 0 and AREA_PRE / AREA_KM2 > MAX_PLACEBO_SHARE:
    PROBLEMS.append(f"the pass before the rain shows {AREA_PRE:.1f} km² ({AREA_PRE / AREA_KM2:.0%} of the flood): the detector finds 'floods' without a flood")
if I_AFTER is None:
    NOTES.append("no later pass on the same orbit: recession check skipped")
elif AREA_AFTER > 1.2 * AREA_KM2:
    NOTES.append(f"the next pass shows more water ({AREA_AFTER:.1f} km²) than the flood pass: check for a second flood (rain returned in early Sep) before posting")
if NEAR_RIVER_SHARE == NEAR_RIVER_SHARE and NEAR_RIVER_SHARE < MIN_NEAR_RIVER_SHARE:
    PROBLEMS.append(f"only {NEAR_RIVER_SHARE:.0%} of the floodwater is within {RIVER_NEAR_KM:g} km of the Vishwamitri (< {MIN_NEAR_RIVER_SHARE:.0%}): likely false water (fields, shadows)")
if NEAR_RIVER_SHARE != NEAR_RIVER_SHARE:
    NOTES.append("Vishwamitri line not found in OSM: river-distance check skipped")
_ch = VARIANTS[VARIANTS.kind == "change"].area_km2
if AREA_KM2 > 0 and len(_ch) and (_ch.max() / max(_ch.min(), 1e-6) > ROBUST_RATIO_MAX ** 2 or _ch.max() / AREA_KM2 > ROBUST_RATIO_MAX or AREA_KM2 / max(_ch.min(), 1e-6) > ROBUST_RATIO_MAX):
    PROBLEMS.append(f"area changes more than ×{ROBUST_RATIO_MAX:g} between the {min(CHANGE_DB_ROBUST):g} and {max(CHANGE_DB_ROBUST):g} dB thresholds: the number is not stable")
_pv = VARIANTS[VARIANTS.kind == "pol"].area_km2
if AREA_KM2 > 0 and len(_pv) and not (0.5 <= float(_pv.iloc[0]) / AREA_KM2 <= 2.0):
    NOTES.append(f"VH gives {float(_pv.iloc[0]):.1f} km² vs {AREA_KM2:.1f} km² in VV (outside ×2): look at both maps")
if N_BLD_SEEN and N_BLD / N_BLD_SEEN > MAX_BLDG_FRAC:
    PROBLEMS.append(f"{N_BLD / N_BLD_SEEN:.0%} of mapped buildings 'in water' (> {MAX_BLDG_FRAC:.0%}): implausible for open-water mapping")
if N_BLD_SEEN < MIN_OSM_BUILDINGS:
    NOTES.append(f"only {N_BLD_SEEN:,} OSM buildings in the box: OSM is incomplete here, building count is a lower bound")
if N_MS_SEEN and N_BLD_SEEN / N_MS_SEEN < OSM_VS_MS_MIN:
    NOTES.append(f"OSM has {N_BLD_SEEN:,} buildings vs {N_MS_SEEN:,} Microsoft footprints: quote the building count as a lower bound")
if SMOKE_TEST:
    PROBLEMS.append("SMOKE TEST MODE: synthetic data")

print("Assumptions:")
for k in ["AOI_BBOX", "GRID_RES_M", "MAIN_POL", "SPECKLE_PX", "CHANGE_DB", "WATER_DB_OFFSET", "MIN_FLOOD_PX",
          "BUILDING_BUFFER_M", "ROAD_BUFFER_M", "DRY_WINDOW", "DANGER_CROSS_IST", "RAIN_START"]:
    print(f"  {k} = {globals()[k]}")
print(f"  water threshold (Otsu) = {WATER_T[MAIN_POL]:.1f} dB · baseline passes = {len(I_DRY)}")
print()
for n in NOTES:
    print("note:", n)
if PROBLEMS:
    print("\n" + "!" * 100)
    for p in PROBLEMS:
        print("!! SANITY CHECK FAILED:", p)
    print("!! Do NOT post these numbers. Fix the cause first.")
    print("!" * 100 + "\n")
else:
    print("All sanity checks passed.\n")

EVENT_DATE = f"{T_EVENT:%d %b %Y}"
HEADLINE = (f"Free Sentinel-1 radar mapped {fmt_km2(AREA_KM2)} km² of open floodwater in and around {CITY} on {EVENT_DATE}, "
            f"{HOURS_AFTER:.0f} hours after the Vishwamitri crossed its danger mark; {N_BLD:,} OpenStreetMap buildings and "
            f"{ROAD_KM:.0f} km of road were in or beside it (estimates).")
print("HEADLINE:", HEADLINE)
print(f"Robustness: area {VARIANTS.area_km2.min():.1f}–{VARIANTS.area_km2.max():.1f} km² across {len(VARIANTS)} variants; "
      f"before the rain {fmt_km2(AREA_PRE)} km²; next pass {fmt_km2(AREA_AFTER)} km²")

In [ ]:
# CELL: results_chart
RADAR_CMAP = matplotlib.colors.LinearSegmentedColormap.from_list("23n_radar", [B.PAPER, B.STONE, B.INK])
_lo, _hi = np.nanpercentile(BASE[MAIN_POL], [2, 99])

def overlay(mask, color, alpha=0.9):
    rgba = np.zeros(mask.shape + (4,), np.float32); rgba[mask] = matplotlib.colors.to_rgba(color, alpha)
    return rgba

def draw_map(ax, img, flood=None, perm=True, river=True, step=1, reveal=1.0, bld=False, flood_color=None):
    ax.imshow(img[::step, ::step], cmap=RADAR_CMAP, vmin=_lo, vmax=_hi, extent=EXTENT_KM, interpolation="nearest")
    if perm:
        ax.imshow(overlay(PERM[::step, ::step], B.AMBER, 0.85), extent=EXTENT_KM, interpolation="nearest")
    if flood is not None:
        f = flood[::step, ::step].copy()
        if reveal < 1:
            f[int(f.shape[0] * reveal):] = False
        ax.imshow(overlay(f, flood_color or B.MOSS, 0.9), extent=EXTENT_KM, interpolation="nearest")
    if river:
        for lon, lat in OSM["river"]:
            r, c = lonlat_to_rc(lon, lat)
            ax.plot(c * GRID_RES_M / 1000, r * GRID_RES_M / 1000, color=B.INK, lw=0.8, ls="--", alpha=0.7)
    if bld:
        r, c = lonlat_to_rc(OSM["bld"][BLD_HIT, 0], OSM["bld"][BLD_HIT, 1])
        ax.scatter(c * GRID_RES_M / 1000, r * GRID_RES_M / 1000, s=0.6, color=B.MOSS, lw=0)
    ax.set_xlim(EXTENT_KM[0], EXTENT_KM[1]); ax.set_ylim(EXTENT_KM[2], EXTENT_KM[3])
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
    for s in ax.spines.values():
        s.set_visible(False)

_tag = " · SMOKE TEST, NOT REAL" if SMOKE_TEST else ""
fig, axes = plt.subplots(1, 2, figsize=(12, 6.6))
draw_map(axes[0], BASE[MAIN_POL], perm=True, step=2)
axes[0].set_title(f"Dry season (median of {len(I_DRY)} passes, Jan–May 2024)", loc="left", fontsize=12)
draw_map(axes[1], EV[MAIN_POL], flood=FLOOD, step=2)
axes[1].set_title(f"{EVENT_DATE} · open floodwater {AREA_KM2:.1f} km²", loc="left", fontsize=12)
fig.suptitle(f"{CITY}: open floodwater from Sentinel-1 radar", x=0.01, ha="left", fontproperties=B.FP.get("display"), fontsize=20)
B.source_note(fig, f"23N Field Test #{FT_NUM} · moss: new open water · amber: dry-season water · dashed: Vishwamitri (OSM){_tag}\n{DATA_CREDIT}")
fig.tight_layout(rect=(0, 0.05, 1, 0.95)); fig.savefig(f"ft{FT_NUM}_flood_map.png", dpi=150); plt.show()

def draw_areas(ax, big=False, grow=1.0):
    labels, vals = [], []
    if I_PRE is not None:
        labels.append(f"Before the rain\n{PASSES.time.iloc[I_PRE]:%d %b}"); vals.append(AREA_PRE)
    labels.append(f"Flood pass\n{T_EVENT:%d %b}"); vals.append(AREA_KM2)
    if I_AFTER is not None:
        labels.append(f"Next pass\n{PASSES.time.iloc[I_AFTER]:%d %b}"); vals.append(AREA_AFTER)
    cols = [B.MOSS if l.startswith("Flood") else B.STONE for l in labels]
    ax.bar(range(len(vals)), [v * grow for v in vals], color=cols, width=0.6)
    for k, v in enumerate(vals):
        ax.text(k, v * grow + max(vals) * 0.03, f"{v:.1f} km²", ha="center", fontsize=16 if big else 11,
                color=B.MOSS if cols[k] == B.MOSS else B.INK)
    ax.set_xticks(range(len(vals))); ax.set_xticklabels(labels, fontsize=14 if big else 10)
    ax.set_ylim(0, max(vals) * 1.25 if max(vals) > 0 else 1); ax.set_ylabel("km² of open water (new)", fontsize=13 if big else 10)
    ax.tick_params(axis="y", labelsize=12 if big else 9)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.6), gridspec_kw=dict(width_ratios=[1, 1.3]))
draw_areas(axes[0]); B.chart_title(axes[0], "Same detector, three passes", "same orbit · same thresholds")
v = VARIANTS.iloc[::-1]
axes[1].barh(v.variant, v.area_km2, color=[B.MOSS if k == "headline" else B.AMBER for k in v.kind])
axes[1].set_xlabel("km² of open floodwater"); B.chart_title(axes[1], "Robustness variants", f"flood pass, {EVENT_DATE}")
B.source_note(fig, f"23N Field Test #{FT_NUM} · {DATA_CREDIT}{_tag}")
fig.tight_layout(rect=(0, 0.04, 1, 1)); fig.savefig(f"ft{FT_NUM}_areas.png", dpi=150); plt.show()

## 5 · Interactive demo
Move the thresholds and see how the flood map and the counts change. The cell doesn't block "Run all"; open the
public `gradio.live` link it prints.

In [ ]:
# CELL: gradio
def demo_fn(change_db, water_offset, pol):
    pol = pol.lower()
    fl, _, _ = detect(EV[pol], BASE[pol], float(change_db), WATER_T[pol] + float(water_offset))
    fb = ndimage.binary_dilation(fl, structure=disk(BUILDING_BUFFER_M / GRID_RES_M))
    nb = int(points_in(fb, OSM["bld"][:, 0], OSM["bld"][:, 1])[0].sum())
    fig, ax = plt.subplots(figsize=(8, 8))
    draw_map(ax, EV[pol], flood=fl, step=2)
    B.source_note(fig, f"{EVENT_DATE} · {pol.upper()} · {DATA_CREDIT}" + (" · SMOKE TEST" if SMOKE_TEST else ""))
    fig.tight_layout()
    return fig, f"{fl.sum() * PIX_KM2:.2f} km² of open floodwater · {nb:,} OSM buildings within {BUILDING_BUFFER_M} m"

try:
    import gradio as gr
    with gr.Blocks(title=f"23N Field Test #{FT_NUM}: flood mapping from radar") as demo:
        gr.Markdown(f"### 23N Field Test #{FT_NUM}: {CITY} flood, {EVENT_DATE}, from Sentinel-1 radar\n"
                    "Independent study on public data. Areas and counts are estimates.")
        s1 = gr.Slider(1, 6, value=CHANGE_DB, step=0.5, label="Must be this many dB darker than the dry season")
        s2 = gr.Slider(-3, 3, value=WATER_DB_OFFSET, step=0.5, label="Shift the water threshold (dB)")
        p = gr.Radio(["VV", "VH"], value=MAIN_POL.upper(), label="Polarisation")
        out_plot, out_txt = gr.Plot(), gr.Textbox(label="Result")
        for c in (s1, s2, p):
            c.change(demo_fn, [s1, s2, p], [out_plot, out_txt])
    demo.launch(share=True, prevent_thread_lock=True)
except Exception as e:
    print("Gradio demo not started:", type(e).__name__, e)

## 6 · Video (1080×1080, 30 fps, H.264) and cover (1080×1350)
Every image and number in the video comes from the run above. `B.Video` saves the file at 1.5× speed by default.

In [ ]:
# CELL: video_lib
VIDEO_PATH, COVER_PATH = f"ft{FT_NUM}_video.mp4", f"ft{FT_NUM}_cover.png"
FPS = 30
VSTEP = max(1, max(H, W) // 600)     # downsample maps for video frames

def _ease(p): p = min(max(p, 0.0), 1.0); return 1 - (1 - p) ** 3
def _prog(t, a, b): return min(max((t - a) / (b - a), 0.0), 1.0)

def _title(fig, title, sub):
    fig.text(0.055, 0.85, title, fontproperties=B.FP.get("display"), fontsize=40, color=B.INK, va="top")
    fig.text(0.055, 0.785, sub.upper(), fontproperties=B.FP.get("mono"), fontsize=14, color=B.MUTED, va="top")

def _map_axes(fig):
    return fig.add_axes([0.13, 0.135, 0.74, 0.62])

def draw_evidence(fig, t):
    # t runs 0..35 inside the evidence section (video seconds 5..40)
    B.header(fig, FT_NUM)
    if t < 8:
        _title(fig, f"{CITY} in the dry season", f"Sentinel-1 VV radar · median of {len(I_DRY)} passes, Jan–May 2024")
        draw_map(_map_axes(fig), BASE[MAIN_POL], step=VSTEP)
        B.caption(fig, "Radar sees through cloud. Dark: buildings. Pale: smooth ground. Amber: water all year")
    elif t < 16:
        _title(fig, f"{EVENT_DATE}, {(T_EVENT + IST):%H:%M} IST", f"Same orbit · {HOURS_AFTER:.0f} h after the river crossed its danger mark")
        draw_map(_map_axes(fig), EV[MAIN_POL], step=VSTEP)
        B.caption(fig, "Calm water reflects the radar away, so it shows up pale")
    elif t < 24:
        g = _ease(_prog(t, 16.5, 22))
        _title(fig, "New open water", f"≥ {CHANGE_DB:g} dB darker than the dry season · dark like water")
        draw_map(_map_axes(fig), EV[MAIN_POL], flood=FLOOD, step=VSTEP, reveal=g)
        fig.text(0.945, 0.785, f"{AREA_KM2 * g:.1f} KM²", fontproperties=B.FP.get("mono_bold"), fontsize=18,
                 color=B.MOSS, ha="right", va="top")
        B.caption(fig, "Moss: open floodwater on the flood pass. Permanent water is left out")
    elif t < 31:
        g = _ease(_prog(t, 24.3, 29))
        _title(fig, "Buildings beside the water", f"OpenStreetMap · water within {BUILDING_BUFFER_M} m")
        ax = _map_axes(fig)
        draw_map(ax, EV[MAIN_POL], flood=FLOOD, step=VSTEP, flood_color=B.MOSS_LIGHT, bld=g > 0.98)
        if g <= 0.98 and N_BLD:
            k = int(N_BLD * g); r, c = lonlat_to_rc(OSM["bld"][BLD_HIT][:k, 0], OSM["bld"][BLD_HIT][:k, 1])
            ax.scatter(c * GRID_RES_M / 1000, r * GRID_RES_M / 1000, s=0.6, color=B.MOSS, lw=0)
        fig.text(0.945, 0.785, f"{int(N_BLD * g):,} BUILDINGS", fontproperties=B.FP.get("mono_bold"), fontsize=18,
                 color=B.MOSS, ha="right", va="top")
        B.caption(fig, f"Plus {ROAD_KM:.0f} km of road through water. Radar misses water in narrow streets")
    else:
        _title(fig, "Same test, three passes", "a pass before the rain should show almost nothing")
        ax = fig.add_axes([0.13, 0.20, 0.82, 0.52])
        draw_areas(ax, big=True, grow=_ease(_prog(t, 31.1, 33)))
        for s in ("top", "right"):
            ax.spines[s].set_visible(False)
        B.caption(fig, "Same orbit, same thresholds. The water recedes by the next pass")

def render_video():
    v = B.Video(VIDEO_PATH, fps=FPS)
    B.hook_card(v.fig, FT_NUM, "How fast can open radar map a city flood?",
                f"{CITY} · August 2024 · Sentinel-1 · free data, free CPU"); v.hold(5)
    step = 2                                          # draw every 2nd frame (15 drawn fps), write each twice
    for k in range(0, 35 * FPS, step):
        v.clear(); draw_evidence(v.fig, k / FPS)
        b = v.grab()
        for _ in range(step):
            v.proc.stdin.write(b); v.frames += 1
    v.clear()
    B.headline_card(v.fig, FT_NUM, f"{AREA_KM2:.1f} km²",
                    f"of open floodwater mapped {HOURS_AFTER:.0f} h after the danger mark, beside {N_BLD:,} buildings.",
                    f"Sentinel-1 VV, {EVENT_DATE} vs dry-season median, ≥{CHANGE_DB:g} dB darker + Otsu water threshold; OSM buildings. Estimate.")
    if SMOKE_TEST:
        v.fig.text(0.5, 0.03, "SMOKE TEST · SYNTHETIC DATA · NOT REAL", ha="center", fontproperties=B.FP.get("mono_bold"), fontsize=14, color=B.AMBER)
    v.hold(12)
    v.clear(); B.end_card(v.fig, FT_NUM, "Data: Copernicus Sentinel-1 (2024) via Planetary Computer · OpenStreetMap (ODbL)"); v.hold(6)
    return v.close()

def render_cover():
    def img(ax):
        draw_map(ax, EV[MAIN_POL], flood=FLOOD, step=max(1, VSTEP // 2))
        # crop a band to the panel's shape (no stretching), centred on the floodwater
        bb = ax.get_position(original=True); fw, fh = ax.figure.get_size_inches()
        wk = EXTENT_KM[1]; hk = min(EXTENT_KM[2], wk * (bb.height * fh) / (bb.width * fw))
        rc = (np.nonzero(FLOOD)[0].mean() * GRID_RES_M / 1000) if FLOOD.any() else EXTENT_KM[2] / 2
        top = min(max(rc - hk / 2, 0), EXTENT_KM[2] - hk)
        ax.set_ylim(top + hk, top); ax.set_aspect("equal", adjustable="box")
    return B.cover(COVER_PATH, FT_NUM, "Mapping a city flood from radar",
                   f"{CITY} · {EVENT_DATE} · Sentinel-1", f"{AREA_KM2:.1f} km²",
                   f"of open floodwater, {HOURS_AFTER:.0f} h after the danger mark" + (" · SMOKE TEST" if SMOKE_TEST else ""),
                   image=img, data_credit="Copernicus Sentinel-1 · © OpenStreetMap")

In [ ]:
# CELL: video_render
render_video()
render_cover()
_probe = subprocess.run(["ffprobe", "-v", "error", "-select_streams", "v:0", "-show_entries",
                         "stream=width,height,r_frame_rate,codec_name:format=duration", "-of", "json", VIDEO_PATH],
                        capture_output=True, text=True)
print(_probe.stdout)
try:
    from google.colab import files
    files.download(VIDEO_PATH); files.download(COVER_PATH)
except Exception:
    print("Not on Colab: video and cover saved in the working directory.")

## 7 · Post text (ready to paste once the sanity checks pass)

In [ ]:
# CELL: post_text
POST_LINKEDIN_TEMPLATE = """How fast can open radar data map a city flood? We tested it on Vadodara, August 2024.

23N Field Test #11: {headline}

On 26 Aug 2024 the Vishwamitri crossed its {danger_ft} ft danger mark in Vadodara after {rain_in} inches of rain in 24 hours. Cloud hides a flood from optical satellites; radar sees through it.

How we tested it:
• Free Copernicus Sentinel-1 radar, no login: the first pass after the danger mark, against a dry-season baseline from the same orbit
• Open water = dark in radar and {change_db} dB darker than the baseline; permanent water left out
• Overlaid on OpenStreetMap buildings and roads

Result: {area} km² from a pass {hours} h after the danger mark. Same test on a pass before the rain: {area_pre} km². Next pass: {area_after} km².

Limits: radar misses water between tall buildings, so dense streets are undercounted; OSM is incomplete; no ground survey. An estimate.

Notebook in the comments.

{signature}
"""
POST_X_TEMPLATE = """How fast can open radar map a city flood? Sentinel-1 over Vadodara, {date}: {area} km² of open floodwater {hours} h after the Vishwamitri crossed its danger mark, touching {n_bld} mapped buildings. 23N Field Test #11.

---
Thread (optional):
1/ Data: Copernicus Sentinel-1 radar via Microsoft Planetary Computer, no login. OpenStreetMap buildings and roads.
2/ Method: dark in radar + {change_db} dB darker than a dry-season baseline from the same orbit. A pass before the rain gives {area_pre} km²; the next pass {area_after} km².
3/ Limits: water between buildings is hard for radar to see, so dense streets are undercounted. Independent study on public data.
"""
def _load_template(name, fallback):
    try:
        return open(name, encoding="utf-8").read()
    except Exception:
        return fallback
POST_VALUES = dict(headline=HEADLINE, danger_ft=DANGER_MARK_FT, rain_in=f"{RAIN_24H_IN:g}", change_db=f"{CHANGE_DB:g}",
                   area=fmt_km2(AREA_KM2), hours=f"{HOURS_AFTER:.0f}", area_pre=fmt_km2(AREA_PRE),
                   area_after=fmt_km2(AREA_AFTER), date=EVENT_DATE, n_bld=f"{N_BLD:,}",
                   signature=B.post_signature(FT_NUM) + " #EarthObservation #FloodMapping #Gujarat")
post_li = _load_template("post_linkedin.md", POST_LINKEDIN_TEMPLATE).format(**POST_VALUES)
post_x = _load_template("post_x.md", POST_X_TEMPLATE).format(**POST_VALUES)
x_main = post_x.split("---")[0].strip()
if PROBLEMS:
    print("!! SANITY CHECKS FAILED (see Results). Do NOT post this text.\n")
print("=" * 30, f"LinkedIn ({len(post_li)} chars)", "=" * 30); print(post_li)
print("=" * 30, f"X (main post {len(x_main)} chars)", "=" * 30); print(post_x)
if len(post_li) > 1300: print(f"WARNING: LinkedIn post is {len(post_li)} chars (> 1,300): shorten before posting")
if len(x_main) > 280: print(f"WARNING: X post is {len(x_main)} chars (> 280): shorten before posting")

## 8 · Limitations (read before quoting the number)

- **Open water, not every wet street.** In dense streets the radar pulse bounces off the water and the walls and
  comes back **bright**, so flooded lanes between buildings are often missed. This undercounts urban flooding; it is
  the main reason the number is "open floodwater", not "flooded area".
- **One snapshot.** Sentinel-1 passed over Vadodara at one moment. The water rose and fell over several days; the
  pass may not be at the peak.
- **"Within hours" is about the acquisition.** The notebook measures hours from the danger mark to the satellite pass.
  How fast the data reached the Planetary Computer is not measured here; Copernicus publishes near-real-time
  products within hours of acquisition.
- **No ground truth.** There is no open flood survey of the city to check against. The checks are a placebo pass,
  a later pass, the distance to the river, and threshold and polarisation variants. NRSC/ISRO's official map of
  28 Aug 2024 (same satellite) is a visual cross-check only; it was issued "for official use" and is not used as data.
- **Dark is not always water.** Smooth tarmac, sand and radar shadow are dark; the dry-season baseline removes most
  of them, not all. Wind or rain at the moment of the pass can roughen water and hide it.
- **Buildings and roads come from OpenStreetMap**, which is incomplete and uneven in Indian cities; the building
  count is a lower bound (the notebook compares it with Microsoft's footprints when it can). A building "beside the
  water" may not have had water inside.
- **The box is ours**, not the municipal boundary.
- **Not a statement about any municipal body, dam authority or agency.** Vadodara is a case study.
- **Licences.** Copernicus Sentinel data terms (free, full and open; "Contains modified Copernicus Sentinel data
  2024"), via Microsoft Planetary Computer, no account. OpenStreetMap © OpenStreetMap contributors, ODbL 1.0.
  Microsoft Global ML Building Footprints: CDLA Permissive 2.0. Code in this repo: see LICENSE.